In [1]:
import numpy as np
import torch
import time
import os
import csv

from datasets import load_dataset, concatenate_datasets
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

from sklearn.metrics import (
    f1_score,
    precision_recall_fscore_support,
    classification_report
)

from transformers import (
    RobertaTokenizerFast,
    RobertaForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    TrainerCallback,
    EarlyStoppingCallback
)

from google.colab import drive


# ============================================================
# Device
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


# ============================================================
# Load Dataset
# ============================================================

train = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="train"
)

val = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="dev"
)

test = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="test"
)

print(train)

train_df = train.to_pandas()
print("Sample data first 5 rows:")
print(train_df.head())


# ============================================================
# Combine and Re-split Dataset into 70 / 20 / 10
# ============================================================

full_data = concatenate_datasets([train, val, test])

len_total = len(full_data)

target_train = int(round(0.70 * len_total))
target_val = int(round(0.20 * len_total))
target_test = len_total - target_train - target_val

split_1 = full_data.train_test_split(train_size=target_train, seed=42)
train_data = split_1["train"]
remaining = split_1["test"]

split_2 = remaining.train_test_split(train_size=target_val, seed=42)
val_data = split_2["train"]
test_data = split_2["test"]

print("Train size:", len(train_data))
print("Validation size:", len(val_data))
print("Test size:", len(test_data))


# ============================================================
# Columns and Labels
# ============================================================

TEXT_COL = "text"

CANDIDATE_EMOTIONS = [
    "anger",
    "fear",
    "joy",
    "sadness",
    "surprise",
    "disgust"
]

EMOTIONS = [
    emotion for emotion in CANDIDATE_EMOTIONS
    if emotion in train_data.column_names
]

print("Used emotion labels:", EMOTIONS)

NUM_LABELS = len(EMOTIONS)
INTENSITY_CLASSES = [0, 1, 2, 3]


# ============================================================
# Tokenizer
# ============================================================

tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


def preprocess(example):
    encoded = tokenizer(
        example[TEXT_COL],
        truncation=True,
        max_length=256
    )

    encoded["labels"] = [
        float(example[emotion]) for emotion in EMOTIONS
    ]

    return encoded


train_tok = train_data.map(preprocess)
val_tok = val_data.map(preprocess)
test_tok = test_data.map(preprocess)

cols = ["input_ids", "attention_mask", "labels"]

train_tok.set_format(type="torch", columns=cols)
val_tok.set_format(type="torch", columns=cols)
test_tok.set_format(type="torch", columns=cols)


# ============================================================
# Metric Helper Functions
# ============================================================

def safe_pearson(x, y):
    try:
        r, _ = pearsonr(x, y)
        return 0.0 if np.isnan(r) else float(r)
    except Exception:
        return 0.0


def convert_regression_to_classes(values):
    """
    Converts regression predictions into intensity classes:
    0 = No emotion
    1 = Slight emotion
    2 = Moderate emotion
    3 = High emotion
    """
    values = np.asarray(values)
    values = np.rint(values)
    values = np.clip(values, 0, 3)

    return values.astype(int)


def compute_metrics(eval_pred):
    preds = eval_pred.predictions
    labels = eval_pred.label_ids

    if isinstance(preds, tuple):
        preds = preds[0]

    preds = np.asarray(preds)
    labels = np.asarray(labels)

    if preds.ndim == 3:
        preds = preds.squeeze(1)

    if labels.ndim == 3:
        labels = labels.squeeze(1)

    pred_classes = convert_regression_to_classes(preds)
    label_classes = convert_regression_to_classes(labels)

    metrics = {}

    # --------------------------------------------------------
    # Pearson correlation per emotion
    # --------------------------------------------------------

    pearson_scores = []

    for i, emotion in enumerate(EMOTIONS):
        pred_col = preds[:, i]
        label_col = labels[:, i]

        r = safe_pearson(pred_col, label_col)

        metrics[f"pearson_{emotion}"] = r
        pearson_scores.append(r)

    metrics["pearson_mean"] = float(np.mean(pearson_scores))

    # --------------------------------------------------------
    # F1 Micro and Macro over all emotions
    # --------------------------------------------------------

    y_true_flat = label_classes.flatten()
    y_pred_flat = pred_classes.flatten()

    metrics["f1_micro"] = f1_score(
        y_true_flat,
        y_pred_flat,
        average="micro",
        labels=INTENSITY_CLASSES,
        zero_division=0
    )

    metrics["f1_macro"] = f1_score(
        y_true_flat,
        y_pred_flat,
        average="macro",
        labels=INTENSITY_CLASSES,
        zero_division=0
    )

    # --------------------------------------------------------
    # F1 per emotion
    # --------------------------------------------------------

    for i, emotion in enumerate(EMOTIONS):
        metrics[f"f1_macro_{emotion}"] = f1_score(
            label_classes[:, i],
            pred_classes[:, i],
            average="macro",
            labels=INTENSITY_CLASSES,
            zero_division=0
        )

        metrics[f"f1_micro_{emotion}"] = f1_score(
            label_classes[:, i],
            pred_classes[:, i],
            average="micro",
            labels=INTENSITY_CLASSES,
            zero_division=0
        )

    return metrics


# ============================================================
# Google Drive and Log File
# ============================================================

drive.mount("/content/drive")

LOG_FILE = "/content/drive/MyDrive/RoBERTa_Log.csv"
os.makedirs("/content/drive/MyDrive", exist_ok=True)


# ============================================================
# Training Settings
# ============================================================

TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16

NUM_EPOCHS = 100

EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_THRESHOLD = 0.0


# ============================================================
# Create CSV Log File
# ============================================================

with open(LOG_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    writer.writerow([
        "section",
        "epoch",
        "train_batch_size",
        "eval_batch_size",
        "train_loss",
        "val_loss",
        "val_f1_micro",
        "val_f1_macro",
        "val_pearson_mean",
        "test_loss",
        "test_f1_micro",
        "test_f1_macro",
        "test_pearson_mean",
        "emotion",
        "class_label",
        "precision",
        "recall",
        "f1_score",
        "support",
        "early_stopping_patience",
        "early_stopping_threshold",
        "best_metric",
        "best_epoch",
        "early_stopped"
    ])


# ============================================================
# Lists for Plotting
# ============================================================

epoch_list = []
train_loss_list = []
val_loss_list = []
val_f1_micro_list = []
val_f1_macro_list = []


# ============================================================
# Callback for Saving Validation Metrics Per Epoch
# ============================================================

class SaveMetricsCallback(TrainerCallback):
    def __init__(self, file_path):
        self.file_path = file_path
        self.current_train_loss = None

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None and "loss" in logs:
            self.current_train_loss = float(logs["loss"])

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is None:
            return

        epoch = metrics.get("epoch", state.epoch)

        if epoch is None:
            epoch = 0

        epoch = round(float(epoch), 4)

        train_loss = self.current_train_loss
        val_loss = float(metrics.get("eval_loss", 0.0))
        val_f1_micro = float(metrics.get("eval_f1_micro", 0.0))
        val_f1_macro = float(metrics.get("eval_f1_macro", 0.0))
        val_pearson_mean = float(metrics.get("eval_pearson_mean", 0.0))

        epoch_list.append(epoch)
        train_loss_list.append(train_loss)
        val_loss_list.append(val_loss)
        val_f1_micro_list.append(val_f1_micro)
        val_f1_macro_list.append(val_f1_macro)

        with open(self.file_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)

            writer.writerow([
                "validation_epoch",
                epoch,
                "",
                "",
                train_loss,
                val_loss,
                val_f1_micro,
                val_f1_macro,
                val_pearson_mean,
                "",
                "",
                "",
                "",
                "",
                "",
                "",
                "",
                "",
                "",
                "",
                "",
                "",
                "",
                ""
            ])


# ============================================================
# Model
# ============================================================

model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=NUM_LABELS,
    problem_type="regression"
).to(device)


# ============================================================
# Training Arguments
# ============================================================

training_args = TrainingArguments(
    output_dir="/content/roberta_output_early_stopping",

    learning_rate=2e-5,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,

    num_train_epochs=NUM_EPOCHS,

    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",

    save_total_limit=1,

    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    report_to="none",

    fp16=torch.cuda.is_available()
)


# ============================================================
# Trainer
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        SaveMetricsCallback(LOG_FILE),
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD
        )
    ]
)


# ============================================================
# Train
# ============================================================

start = time.time()

train_result = trainer.train()

end = time.time()

total_training_time = end - start
completed_epochs = trainer.state.epoch

print(f"Total training time: {total_training_time:.1f} seconds")

if completed_epochs is not None and completed_epochs > 0:
    print(f"Average time per epoch: {total_training_time / completed_epochs:.1f} seconds")

print("Best metric:", trainer.state.best_metric)
print("Best model checkpoint:", trainer.state.best_model_checkpoint)
print("Log file saved at:", LOG_FILE)


# ============================================================
# Save Early Stopping and Batch Size Summary
# Batch size is written only once here
# ============================================================

early_stopped = False

if completed_epochs is not None:
    early_stopped = completed_epochs < NUM_EPOCHS

best_epoch = ""

if trainer.state.best_model_checkpoint is not None:
    try:
        best_epoch = trainer.state.best_model_checkpoint.split("-")[-1]
    except Exception:
        best_epoch = ""

with open(LOG_FILE, "a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    writer.writerow([
        "training_summary",
        completed_epochs,
        TRAIN_BATCH_SIZE,
        EVAL_BATCH_SIZE,
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        EARLY_STOPPING_PATIENCE,
        EARLY_STOPPING_THRESHOLD,
        trainer.state.best_metric,
        best_epoch,
        early_stopped
    ])


# ============================================================
# Final Validation Evaluation Using Best Model
# ============================================================

final_val_metrics = trainer.evaluate(eval_dataset=val_tok)

print("Final validation metrics:")
print(final_val_metrics)


# ============================================================
# Final Test Evaluation Using Best Model
# ============================================================

test_metrics = trainer.evaluate(eval_dataset=test_tok)

print("Final test metrics:")
print(test_metrics)

with open(LOG_FILE, "a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    writer.writerow([
        "final_test_summary",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        float(test_metrics.get("eval_loss", 0.0)),
        float(test_metrics.get("eval_f1_micro", 0.0)),
        float(test_metrics.get("eval_f1_macro", 0.0)),
        float(test_metrics.get("eval_pearson_mean", 0.0)),
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        ""
    ])


# ============================================================
# Get Predictions for Classwise Test Results
# ============================================================

test_predictions = trainer.predict(test_tok)

preds = test_predictions.predictions
labels = test_predictions.label_ids

if isinstance(preds, tuple):
    preds = preds[0]

preds = np.asarray(preds)
labels = np.asarray(labels)

if preds.ndim == 3:
    preds = preds.squeeze(1)

if labels.ndim == 3:
    labels = labels.squeeze(1)

pred_classes = convert_regression_to_classes(preds)
label_classes = convert_regression_to_classes(labels)


# ============================================================
# Test Classwise Results Per Emotion
# ============================================================

with open(LOG_FILE, "a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    for i, emotion in enumerate(EMOTIONS):
        y_true = label_classes[:, i]
        y_pred = pred_classes[:, i]

        precision, recall, f1, support = precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=INTENSITY_CLASSES,
            zero_division=0
        )

        for class_index, class_label in enumerate(INTENSITY_CLASSES):
            writer.writerow([
                "test_classwise_per_emotion",
                "",
                "",
                "",
                "",
                "",
                "",
                "",
                "",
                "",
                "",
                "",
                "",
                emotion,
                class_label,
                float(precision[class_index]),
                float(recall[class_index]),
                float(f1[class_index]),
                int(support[class_index]),
                "",
                "",
                "",
                "",
                ""
            ])


# ============================================================
# Overall Test Classwise Results Across All Emotions
# ============================================================

y_true_flat = label_classes.flatten()
y_pred_flat = pred_classes.flatten()

precision, recall, f1, support = precision_recall_fscore_support(
    y_true_flat,
    y_pred_flat,
    labels=INTENSITY_CLASSES,
    zero_division=0
)

with open(LOG_FILE, "a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    for class_index, class_label in enumerate(INTENSITY_CLASSES):
        writer.writerow([
            "test_classwise_overall",
            "",
            "",
            "",
            "",
            "",
            "",
            "",
            "",
            "",
            "",
            "",
            "",
            "all_emotions",
            class_label,
            float(precision[class_index]),
            float(recall[class_index]),
            float(f1[class_index]),
            int(support[class_index]),
            "",
            "",
            "",
            "",
            ""
        ])


# ============================================================
# Print Classification Reports
# ============================================================

print("\nOverall Test Classification Report Across All Emotions:")
print(
    classification_report(
        y_true_flat,
        y_pred_flat,
        labels=INTENSITY_CLASSES,
        zero_division=0
    )
)

print("\nClasswise Test Reports Per Emotion:")

for i, emotion in enumerate(EMOTIONS):
    print(f"\nEmotion: {emotion}")
    print(
        classification_report(
            label_classes[:, i],
            pred_classes[:, i],
            labels=INTENSITY_CLASSES,
            zero_division=0
        )
    )


# ============================================================
# Save Best Model and Tokenizer
# ============================================================

BEST_MODEL_DIR = "/content/drive/MyDrive/RoBERTa_Best_Model"

trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)

print("Best model saved at:", BEST_MODEL_DIR)
print("Final log file saved at:", LOG_FILE)


# ============================================================
# Graph 1: Training Loss vs Epoch
# ============================================================

plt.figure(figsize=(8, 5))
plt.plot(epoch_list, train_loss_list, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss vs Epoch")
plt.grid(True)
plt.show()


# ============================================================
# Graph 2: Validation Loss vs Epoch
# ============================================================

plt.figure(figsize=(8, 5))
plt.plot(epoch_list, val_loss_list, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.title("Validation Loss vs Epoch")
plt.grid(True)
plt.show()


# ============================================================
# Graph 3: Validation F1 Micro vs Epoch
# ============================================================

plt.figure(figsize=(8, 5))
plt.plot(epoch_list, val_f1_micro_list, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Validation F1 Micro")
plt.title("Validation F1 Micro vs Epoch")
plt.grid(True)
plt.show()


# ============================================================
# Graph 4: Validation F1 Macro vs Epoch
# ============================================================

plt.figure(figsize=(8, 5))
plt.plot(epoch_list, val_f1_macro_list, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Validation F1 Macro")
plt.title("Validation F1 Macro vs Epoch")
plt.grid(True)
plt.show()

KeyboardInterrupt: 

In [ ]:
test_results = trainer.evaluate(test_tok)
print("Test results:", test_results)

In [ ]:
def predict_intensities(text: str):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits.detach().cpu().numpy()[0]

    discrete = np.clip(np.rint(logits), 0, 3).astype(int)

    return {
        EMOTIONS[i]: {
            "raw": float(logits[i]),
            "intensity_0_3": int(discrete[i])
        }
        for i in range(len(EMOTIONS))
    }

print(predict_intensities("I feel so happy and joyful today!"))